# 07. Multivariate Survival Analysis & Diagnostics
This notebook evaluates the independent prognostic value of the 5-gene signature alongside clinical covariates and performs statistical diagnostics (VIF) and visualization (Forest Plot).

**Revision Highlights:**
1. **Multivariate Cox Model**: Adjusted for Age, Grade, ER Status, and Lymph Node Status.
2. **Collinearity Diagnostics**: Calculated Variance Inflation Factor (VIF).
3. **Publication-Ready Forest Plot**: High-resolution visualization of hazard ratios.

In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import matplotlib.pyplot as plt
import os

MATRIX_PATH = os.path.join("..", "raw_data", "GSE42568_series_matrix.txt")
GPL_PATH = os.path.join("..", "raw_data", "GPL570-55999.txt")
SIGNATURE_GENES = ['FHL1', 'PCK1', 'ACVR1C', 'FABP4', 'MT1H']
FIGURE_DIR = "../figures"

def load_geo_clinical(file_path):
    clinical_data = {}
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith("!Sample_characteristics_ch1"):
                parts = line.split("\t")
                key_raw = parts[1].split(":")[0].strip().replace('"', '')
                values = [p.split(":")[-1].strip().replace('"', '') for p in parts[1:]]
                clinical_data[key_raw] = values
            if line.startswith("!series_matrix_table_begin"):
                break
    return pd.DataFrame(clinical_data)

df_clinical = load_geo_clinical(MATRIX_PATH)
print(f"Loaded clinical data for {len(df_clinical)} samples.")

In [ ]:
def load_expression_subset(matrix_path, genes, annotation_path):
    gpl = pd.read_csv(annotation_path, sep="\t", comment="#", low_memory=False)
    gene_to_probes = gpl[gpl['Gene Symbol'].isin(genes)][['ID', 'Gene Symbol']]
    skip = 0
    with open(matrix_path, 'r') as f:
        for i, line in enumerate(f):
            if line.startswith("!series_matrix_table_begin"):
                skip = i + 1
                break
    expr = pd.read_csv(matrix_path, sep="\t", skiprows=skip, nrows=None, comment="!")
    expr = expr.rename(columns={'ID_REF': 'ID'})
    merged = expr.merge(gene_to_probes, on='ID')
    gene_expr = merged.groupby('Gene Symbol').mean(numeric_only=True).T
    return gene_expr

df_expr = load_expression_subset(MATRIX_PATH, SIGNATURE_GENES, GPL_PATH)
df_expr_z = (df_expr - df_expr.mean()) / df_expr.std()
df_expr['Signature_Score'] = df_expr_z.mean(axis=1)

gsm_ids = []
with open(MATRIX_PATH, 'r') as f:
    for line in f:
        if line.startswith("!Series_sample_id"):
            gsm_ids = line.split("\t")[1].strip().replace('"', '').split()
            break

df_clinical['GSM'] = gsm_ids
df_merged = df_clinical.merge(df_expr[['Signature_Score']], left_on='GSM', right_index=True)

df_final = df_merged.rename(columns={
    'relapse free survival time_days': 'Time',
    'relapse free survival event': 'Event',
    'age': 'Age',
    'er_status': 'ER_Status',
    'grade': 'Tumor_Grade',
    'lymph node status': 'Lymph_Node_Status'
})
df_final = df_final[df_final['tissue'] == 'breast cancer'].copy()
df_final = df_final[['Time', 'Event', 'Age', 'ER_Status', 'Tumor_Grade', 'Lymph_Node_Status', 'Signature_Score']]
for col in df_final.columns:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
df_final = df_final.dropna()

print(f"Cleaned data for {len(df_final)} patient samples.")

### 1. Collinearity Diagnostics (VIF)
A Variance Inflation Factor (VIF) > 5.0 would indicate high redundancy. Here we check the orthogonality of the signature.

In [ ]:
print("--- Variance Inflation Factor Diagnostics ---")
X_vif = df_final[['Signature_Score', 'Lymph_Node_Status', 'ER_Status', 'Age', 'Tumor_Grade']].dropna()
X_vif = add_constant(X_vif)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
display(vif_data.loc[vif_data['Variable'] != 'const'])

### 2. Multivariate Cox Proportional Hazards Model

In [ ]:
cph = CoxPHFitter()
cph.fit(df_final, duration_col='Time', event_col='Event')
summary = cph.summary
display(summary)

### 3. Publication-Ready Forest Plot

In [ ]:
variables = ['Age', 'Tumor_Grade', 'Signature_Score', 'ER_Status', 'Lymph_Node_Status']
hr = [summary.loc[v, 'exp(coef)'] for v in variables]
ci_lower = [np.exp(summary.loc[v, 'coef lower 95%']) for v in variables]
ci_upper = [np.exp(summary.loc[v, 'coef upper 95%']) for v in variables]
p_values = [summary.loc[v, 'p'] for v in variables]

plt.figure(figsize=(10, 5), dpi=300)
y_pos = np.arange(len(variables))
error_left = np.array(hr) - np.array(ci_lower)
error_right = np.array(ci_upper) - np.array(hr)
colors = ['#4c72b0' if p > 0.05 else '#c44e52' for p in p_values]

plt.errorbar(hr, y_pos, xerr=[error_left, error_right], fmt='o', color='black', 
             ecolor='gray', elinewidth=2, capsize=5, markersize=8, label='Hazard Ratio (95% CI)')

for i, color in enumerate(colors):
    plt.plot(hr[i], y_pos[i], 'o', color=color, markersize=8)

plt.axvline(x=1.0, color='black', linestyle='--', linewidth=1.2)
plt.xscale('log')
plt.xlim(0.1, 20.0)
plt.xticks([0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0], 
           ['0.1', '0.2', '0.5', '1.0', '2.0', '5.0', '10.0', '20.0'])
plt.yticks(y_pos, [v.replace('_', ' ') for v in variables], fontsize=12, weight='bold')
plt.xlabel("Hazard Ratio (Log Scale)", fontsize=12, labelpad=10)
plt.title("Multivariate Cox Proportional Hazards Model (GSE42568)", fontsize=14, pad=15, weight='bold')

for i, p in enumerate(p_values):
    plt.text(22.0, y_pos[i], f"p = {p:.3f}" if p >= 0.001 else "p < 0.001", 
             va='center', fontsize=11, weight='bold' if p < 0.05 else 'normal')

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "Multivariate_Survival_Forest_Plot.jpeg"), dpi=300, bbox_inches='tight')
plt.show()